# Evaluation: Base Llama 3.1 8B Instruct vs. Fine-tuned (step-1000)

Runs both the **base** Llama 3.1 8B Instruct and the **fine-tuned LoRA at step 1000** (best val-loss checkpoint) on `test.jsonl` and reports:

1. **Loss** — full-sequence cross-entropy on the gold chat (directly comparable to the val-loss curve from training).
2. **Output quality** — JSON parse rate, `type` exact match, `risk` exact match.
3. **Side-by-side examples** — eyeball the difference.
4. **Optional Gemini LLM-as-judge** — semantic quality of `summary` / `reason` / `suggestion`.

**Runtime:** Colab T4 GPU. ~30–45 min total for 400 test examples (loss + greedy generation, both models).

**Before running:** Drive must contain `MyDrive/legal-clauses/test.jsonl` and one of:
- `MyDrive/legal-clauses/training-outputs/checkpoint-1000/` (preferred — explicit step-1000), OR
- `MyDrive/legal-clauses/legal-llama3-qlora/` (the saved best adapter — should also be step 1000 since `load_best_model_at_end=True`).

The notebook auto-detects which one is available and prints the resolved step number from `trainer_state.json` to confirm.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -U transformers peft accelerate bitsandbytes datasets google-generativeai

## 2. Imports & Paths

In [ ]:
import os
import gc
import json
import re
import time
import random
from pathlib import Path

import torch
from tqdm.auto import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MAX_SEQ_LEN = 2048
MAX_NEW_TOKENS = 512

DATA_DIR = "/content/drive/MyDrive/legal-clauses"
TRAINING_OUT = f"{DATA_DIR}/training-outputs"
ADAPTER_FALLBACK = f"{DATA_DIR}/legal-llama3-qlora"
EVAL_OUT = f"{DATA_DIR}/eval-outputs"

SYSTEM_PROMPT = """You are a legal clause analyst. Given a legal clause, analyze it and respond with a JSON object containing:
- "type": the clause type
- "summary": a 1-2 sentence summary
- "risk": "Low", "Medium", or "High"
- "reason": a 1-2 sentence risk justification
- "suggestion": a negotiation or improvement suggestion"""

## 3. Mount Drive & Load Test Set

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Path(EVAL_OUT).mkdir(parents=True, exist_ok=True)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

test_set = load_jsonl(f"{DATA_DIR}/test.jsonl")
print(f"Loaded {len(test_set)} test examples")
print(f"Sample input: {test_set[0]['input'][:200]}...")
print(f"Sample output: {test_set[0]['output']}")

## 4. Helper Functions

- `compute_loss` — full-sequence CE on system+user+assistant. Matches what `SFTTrainer` reported as `eval_loss` during training, so the numbers are directly comparable to the 0.845 plateau.
- `generate_response` — greedy decoding (`do_sample=False`) for reproducibility.
- `parse_json` — strips markdown fences and extracts the first JSON object.

In [ ]:
def build_chat_with_target(tokenizer, example):
    output = example["output"]
    if isinstance(output, str):
        output = json.loads(output)
    output_str = json.dumps(output, indent=2)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": output_str},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def build_chat_for_inference(tokenizer, example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["input"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )


def compute_loss(model, tokenizer, example):
    text = build_chat_with_target(tokenizer, example)
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)
    ids = enc.input_ids.to(model.device)
    with torch.no_grad():
        out = model(input_ids=ids, labels=ids)
    return out.loss.item()


def generate_response(model, tokenizer, example):
    ids = build_chat_for_inference(tokenizer, example).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)


def parse_json(text):
    if not text:
        return None
    s = text.strip()
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    try:
        return json.loads(s)
    except Exception:
        m = re.search(r"\{[\s\S]*\}", s)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
        return None


def run_eval(model, tokenizer, dataset, output_path, label):
    records = []
    for ex in tqdm(dataset, desc=label):
        loss = None
        try:
            loss = compute_loss(model, tokenizer, ex)
        except Exception as e:
            print(f"loss err: {e}")
        generated = ""
        try:
            generated = generate_response(model, tokenizer, ex)
        except Exception as e:
            print(f"gen err: {e}")
        gold = ex["output"]
        if isinstance(gold, str):
            gold = json.loads(gold)
        records.append({
            "input": ex["input"],
            "gold": gold,
            "generated": generated,
            "loss": loss,
        })
    with open(output_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"Wrote {len(records)} records → {output_path}")
    return records

## 5. Load Base Model (4-bit)

Same quantization config as training so loss numbers are comparable.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
)
base_model.config.use_cache = True
base_model.eval()

print(f"Loaded {MODEL_NAME}")
print(f"GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 6. Phase A — Evaluate Base Model on Test Set

In [ ]:
base_records = run_eval(
    base_model, tokenizer, test_set,
    output_path=f"{EVAL_OUT}/base_test_outputs.jsonl",
    label="base",
)

## 7. Attach LoRA Adapter (step-1000 checkpoint)

We **add** the LoRA on top of the already-loaded base model — no need to reload base.

In [ ]:
# Resolve which adapter to use: prefer explicit checkpoint-1000.
explicit_ckpt = f"{TRAINING_OUT}/checkpoint-1000"
if os.path.isdir(explicit_ckpt):
    adapter_path = explicit_ckpt
elif os.path.isdir(ADAPTER_FALLBACK):
    adapter_path = ADAPTER_FALLBACK
    print("NOTE: checkpoint-1000 not found — falling back to the saved best adapter.")
    print("      Since the training run used load_best_model_at_end=True, this should still be step-1000.")
else:
    raise FileNotFoundError(
        f"Neither {explicit_ckpt} nor {ADAPTER_FALLBACK} exists. "
        f"Confirm your training outputs are on Drive."
    )

# Confirm step number from trainer_state.json if available
state_path = f"{adapter_path}/trainer_state.json"
if os.path.isfile(state_path):
    with open(state_path) as f:
        state = json.load(f)
    print(f"Adapter:        {adapter_path}")
    print(f"global_step:    {state.get('global_step')}")
    print(f"best_metric:    {state.get('best_metric')}")
    print(f"best_ckpt path: {state.get('best_model_checkpoint')}")
else:
    print(f"Adapter: {adapter_path} (no trainer_state.json — can't confirm step)")

In [ ]:
from peft import PeftModel

ft_model = PeftModel.from_pretrained(base_model, adapter_path)
ft_model.eval()
print(f"GPU mem after LoRA load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 8. Phase B — Evaluate Fine-tuned Model on Test Set

In [ ]:
ft_records = run_eval(
    ft_model, tokenizer, test_set,
    output_path=f"{EVAL_OUT}/ft_test_outputs.jsonl",
    label="fine-tuned (step-1000)",
)

## 9. Metrics Comparison

In [ ]:
def norm(s):
    return str(s or "").strip().lower()

def compute_metrics(records):
    n = len(records)
    parsed = type_match = risk_match = 0
    losses = [r["loss"] for r in records if r["loss"] is not None]
    for r in records:
        out = parse_json(r["generated"])
        if not isinstance(out, dict):
            continue
        parsed += 1
        gold = r["gold"] if isinstance(r["gold"], dict) else {}
        if norm(out.get("type")) == norm(gold.get("type")):
            type_match += 1
        if norm(out.get("risk")) == norm(gold.get("risk")):
            risk_match += 1
    return {
        "n": n,
        "mean_loss": sum(losses) / len(losses) if losses else None,
        "json_parse_rate": parsed / n,
        "type_exact_match": type_match / n,
        "risk_exact_match": risk_match / n,
    }

base_metrics = compute_metrics(base_records)
ft_metrics = compute_metrics(ft_records)

print(f"{'Metric':<22} {'Base':>12} {'Fine-tuned':>14} {'Delta':>12}")
print("-" * 62)
for k in ["n", "mean_loss", "json_parse_rate", "type_exact_match", "risk_exact_match"]:
    b, f = base_metrics[k], ft_metrics[k]
    if isinstance(b, float) and isinstance(f, float):
        delta = f"{f - b:+.4f}"
        print(f"{k:<22} {b:>12.4f} {f:>14.4f} {delta:>12}")
    else:
        print(f"{k:<22} {str(b):>12} {str(f):>14}")

with open(f"{EVAL_OUT}/comparison_summary.json", "w") as fp:
    json.dump({"base": base_metrics, "fine_tuned": ft_metrics}, fp, indent=2)

## 10. Side-by-side Examples

Eyeball the difference. The base model often emits prose or JSON-with-prose; the fine-tuned one should emit clean JSON.

In [ ]:
random.seed(42)
indices = random.sample(range(len(test_set)), k=min(10, len(test_set)))

for i in indices:
    print("=" * 80)
    print(f"# Example {i}")
    print("-- INPUT --")
    print(base_records[i]["input"][:400] + ("..." if len(base_records[i]["input"]) > 400 else ""))
    print("-- GOLD --")
    print(json.dumps(base_records[i]["gold"], indent=2)[:400])
    print("-- BASE --")
    print(base_records[i]["generated"][:600])
    print("-- FINE-TUNED --")
    print(ft_records[i]["generated"][:600])
    print()

## 11. (Optional) Gemini LLM-as-Judge

Pairwise comparison on a 50-example sample for the free-text fields (`summary`, `reason`, `suggestion`). Position is randomized per example to control for position bias.

**Caveat:** The training labels were also Gemini-generated, so this judge is biased toward outputs that mimic Gemini's style. Read the win-rate as 'closer to the distillation target', not 'objectively better'.

In [ ]:
from google.colab import userdata
import google.generativeai as genai

# Set GEMINI_API_KEY via Colab Secrets (left sidebar → Secrets) or paste here.
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") if "GEMINI_API_KEY" in userdata.keys() else None
if not GEMINI_API_KEY:
    GEMINI_API_KEY = ""  # paste here if not using Secrets

genai.configure(api_key=GEMINI_API_KEY)
judge = genai.GenerativeModel("gemini-1.5-flash")

JUDGE_PROMPT = """You are evaluating two AI assistant responses to a legal-clause analysis prompt.
Both responses should produce JSON with type/summary/risk/reason/suggestion. Judge ONLY the
free-text fields (summary, reason, suggestion). Ignore JSON formatting differences and the
'type' / 'risk' categorical fields.

Choose the response that is more accurate, specific to the clause, and useful as legal commentary.
If they are roughly equivalent, answer TIE.

Reply with exactly one token: A, B, or TIE.

CLAUSE:
{clause}

RESPONSE A:
{a}

RESPONSE B:
{b}

Verdict (A / B / TIE):"""

def judge_pair(clause, base_out, ft_out):
    flip = random.random() < 0.5
    a, b = (ft_out, base_out) if flip else (base_out, ft_out)
    prompt = JUDGE_PROMPT.format(clause=clause[:3000], a=a[:2000], b=b[:2000])
    try:
        resp = judge.generate_content(prompt).text.strip().upper()
    except Exception as e:
        return "ERR"
    if "TIE" in resp:
        return "tie"
    if resp.startswith("A"):
        return "ft" if flip else "base"
    if resp.startswith("B"):
        return "base" if flip else "ft"
    return "unknown"

random.seed(123)
sample_idx = random.sample(range(len(test_set)), k=min(50, len(test_set)))

verdicts = {"base": 0, "ft": 0, "tie": 0, "unknown": 0, "ERR": 0}
judge_records = []
for i in tqdm(sample_idx, desc="judge"):
    v = judge_pair(
        base_records[i]["input"],
        base_records[i]["generated"],
        ft_records[i]["generated"],
    )
    verdicts[v] = verdicts.get(v, 0) + 1
    judge_records.append({"idx": i, "verdict": v})
    time.sleep(2)  # respect Gemini rate limits

print("\nLLM-as-judge verdicts (n=50):")
for k, v in verdicts.items():
    print(f"  {k:<10} {v}")

with open(f"{EVAL_OUT}/judge_results.json", "w") as fp:
    json.dump({"verdicts": verdicts, "records": judge_records}, fp, indent=2)

## 12. Interpreting Results

**What to expect:**

- **Loss:** Base will be substantially higher than fine-tuned (likely 1.5–2.5+ vs ~0.85). Base hasn't been trained on this specific JSON output format, so it pays a heavy CE penalty on the gold tokens.
- **JSON parse rate:** Base often emits prose-with-JSON or markdown-fenced JSON; fine-tuned should be near 100%. The `parse_json` helper strips fences, but free-form prose still won't parse.
- **Type / risk exact match:** Even when base parses, it tends to use different label vocabularies (e.g., 'Assignment' vs 'Anti-Assignment'). The fine-tuned model has memorized the LEDGAR/CUAD label space.
- **LLM judge:** Useful as a tiebreaker on free-text quality. Take with a grain of salt — your gold labels are also Gemini-generated, so the judge has a stylistic prior toward fine-tuned outputs.

**On the overfitting from your loss curve:** Val loss plateaued at 0.845 from step 1000. The test loss this notebook reports should be in the same ballpark (~0.83–0.86) if val and test are drawn from the same distribution. If test loss is much higher, you have a generalization problem beyond just overfit. If it's similar, the fine-tuned model is doing what it claims — the plateau just means there's no value in training past step 1000.